In [6]:
%idle_timeout 2880
%glue_version 5.1
%worker_type G.1X
%number_of_workers 5

import sys
from awsglue.transforms import *
from awsglue.utils import getResolvedOptions
from pyspark.context import SparkContext
from awsglue.context import GlueContext
from awsglue.job import Job
from pyspark.sql.functions import explode, col, to_date, lit
from pyspark.sql.types import IntegerType

from datetime import datetime
from awsglue.dynamicframe import DynamicFrame

import boto3
  
sc = SparkContext.getOrCreate()
glueContext = GlueContext(sc)
spark = glueContext.spark_session
job = Job(glueContext)

You are already connected to a glueetl session deac3b4b-0344-466b-b81c-6a336849d61f.

No change will be made to the current session that is set as glueetl. The session configuration change will apply to newly created sessions.


Current idle_timeout is None minutes.
idle_timeout has been set to 2880 minutes.


You are already connected to a glueetl session deac3b4b-0344-466b-b81c-6a336849d61f.

No change will be made to the current session that is set as glueetl. The session configuration change will apply to newly created sessions.


Setting Glue version to: 5.1


You are already connected to a glueetl session deac3b4b-0344-466b-b81c-6a336849d61f.

No change will be made to the current session that is set as glueetl. The session configuration change will apply to newly created sessions.


Previous worker type: None
Setting new worker type to: G.1X


You are already connected to a glueetl session deac3b4b-0344-466b-b81c-6a336849d61f.

No change will be made to the current session that is set as glueetl. The session configuration change will apply to newly created sessions.


Previous number of workers: None
Setting new number of workers to: 5


In [2]:
s3_path = "s3://spotify-etl-project-204537390950-us-east-1-an/raw_data/to_processed/"
source_df = glueContext.create_dynamic_frame_from_options(
    connection_type="s3",
    connection_options={"paths":[s3_path]},
    format="json"
)

In [3]:
spotify_df = source_df.toDF()

/usr/lib/spark/python/lib/pyspark.zip/pyspark/sql/dataframe.py:147: UserWarning: DataFrame constructor is internal. Do not directly use it.


In [12]:
def process_albums(df):
    df = df.withColumn("items", explode("items")).select(
        col("items.item.album.id").alias("album_id"),
        col("items.item.album.name").alias("album_name"),
        col("items.item.album.release_date").alias("release_date"),
        col("items.item.album.total_tracks").alias("total_tracks"),
        col("items.item.album.external_urls.spotify").alias("url")
    ).drop_duplicates(["album_id"])
    
    return df

In [7]:
def process_artists(df):
    df = df.withColumn("items", explode(col("items"))).withColumn("artists", explode(col("items.item.artists"))).select(
        col("artists.id").alias("artist_id"),
        col("artists.name").alias("artist_name"),
        col("artists.external_urls.spotify").alias("external_url")
    ).drop_duplicates(["artist_id"])
    
    return df

In [9]:
def process_songs(df):
    item_fields = [f.name for f in df.schema["items"].dataType.elementType["item"].dataType.fields]
    popularity_col = col("items.item.popularity") if "popularity" in item_fields else lit(None).cast(IntegerType())
    df = df.withColumn("items", explode(col("items"))).withColumn("artists", explode(col("items.item.artists"))).select(
        col("items.item.id").alias("song_id"),
        col("items.item.name").alias("song_name"),
        col("items.item.duration_ms").alias("duration_ms"),
        col("items.item.external_urls.spotify").alias("url"),
        popularity_col.alias("popularity"),
        col("items.added_at").alias("song_added"),
        col("items.item.album.id").alias("album_id"),
        col("artists.id").alias("artist_id")
    ).drop_duplicates(["song_id"])
    
    df = df.withColumn("song_added", to_date(col("song_added")))
    
    return df


In [13]:
#process data
album_df = process_albums(spotify_df)
artist_df = process_artists(spotify_df)
song_df = process_songs(spotify_df)

In [18]:
def write_to_s3(df, path_suffix, format_type="csv"):
    # Convert back to DynamicFrame
    dynamic_frame = DynamicFrame.fromDF(df, glueContext, "dynamic_frame")
    
    glueContext.write_dynamic_frame.from_options(
        frame = dynamic_frame,
        connection_type = "s3",
        connection_options = {"path": f"s3://spotify-etl-project-204537390950-us-east-1-an/transformed_data/{path_suffix}/"},
        format = format_type
    )

In [19]:
#write data to s3   
write_to_s3(album_df, "album_data/album_transformed_{}".format(datetime.now().strftime("%Y-%m-%d")), "csv")
write_to_s3(artist_df, "artist_data/artist_transformed_{}".format(datetime.now().strftime("%Y-%m-%d")), "csv")
write_to_s3(song_df, "songs_data/songs_transformed_{}".format(datetime.now().strftime("%Y-%m-%d")), "csv")

In [7]:
#copy processed raw data into raw/processed folder, and delete the original raw data in raw/to_processed
def list_s3_objects(bucket, prefix):
    s3_client = boto3.client('s3')
    response = s3_client.list_objects_v2(Bucket=bucket, Prefix=prefix)
    keys = [content['Key'] for content in response.get('Contents', []) if content['Key'].endswith('json')]
    return keys

bucket_name = "spotify-etl-project-204537390950-us-east-1-an"
prefix = "raw_data/to_processed/"
spotify_keys = list_s3_objects(bucket_name, prefix)


In [1]:
def move_and_delete_files(spotify_keys, Bucket):
    s3_source = boto3.resource('s3')
    for key in spotify_keys:
        copy_source = {
            'Bucket': Bucket,
            'Key': key
        }
        #Define the destinationkey
        destination_key = 'raw_data/processed/' + key.split("/")[-1]
        # Move file to processed folder
        s3_source.meta.client.copy(copy_source, Bucket, destination_key)
        # Delete the original file
        s3_source.Object(Bucket, key).delete()

move_and_delete_files(spotify_keys, bucket_name)

NameError: name 'spotify_keys' is not defined


In [ ]:
job.commit()